# Orchestrator-Workers: el manager decide sobre la marcha

Clasificación: **Proceso jerárquico.** Un manager LLM lee la petición del cliente y decide qué agentes necesita, en qué orden y cuánto trabajo darle a cada uno.

A diferencia de parallelization (donde las 4 tasks están fijadas antes de arrancar), aquí el manager adapta la ejecución al caso concreto. Un cliente que solo quiere relajarse necesita más trabajo de actividades; uno con itinerario ajustado, más de vuelos.

## Cómo funciona en CrewAI

`Process.hierarchical` activa un manager automático que orquesta a los agentes. El manager recibe la task principal, decide a quién delegar, y puede volver a consultar al mismo agente si necesita más detalle.

```python
crew = Crew(
    agents=[...],
    tasks=[main_task],
    process=Process.hierarchical,
    manager_llm="gpt-4o-mini",
)
```

In [1]:
!uv pip install -r requirements.txt --quiet

In [2]:
from dotenv import load_dotenv
import nest_asyncio

load_dotenv()
nest_asyncio.apply()

In [4]:
from crewai import Agent, Task, Crew, Process
from viajes_crew import ViajesCrew

base_crew = ViajesCrew()

peticion_cliente = (
    "Viaje a Islandia, 5 dias, 2 personas, 2200 EUR en total. "
    "Lo unico que de verdad importa es ver auroras boreales y banarnos en fuentes termales; "
    "el resto (vuelos, alojamiento) lo he revisado ya manualmente."
)

manager = Agent(
    role="LLM Manager",
    goal="Decide which specialized agents should be invoked to deliver the user's request.",
    backstory="You are a manager agent in an Orchestrator-Worker setup. You're responsible for deciding the most suitable agents for a particular request",
    llm="gpt-4o-mini",
    verbose=True
)

main_task = Task(
    description=(
        f"Peticion del cliente: {peticion_cliente}\n\n"
        "Decide que especialistas necesitas y en que orden, dado las prioridades del cliente. "
        "Entrega un itinerario completo y el presupuesto desglosado."
    ),
    expected_output="Itinerario dia a dia con vuelos, alojamiento, actividades y transporte, coste por partida y total dentro del presupuesto indicado.",
    agent=manager,
)

crew = Crew(
    agents=[base_crew.vuelos(), base_crew.alojamiento(), base_crew.actividades(), base_crew.transporte()],
    tasks=[main_task],
    process=Process.hierarchical,
    manager_llm="gpt-4o-mini",
    verbose=True,
)

result = await crew.kickoff_async()
print(result.raw)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.14                                                                                       │
│  Latest version:  1.15.16                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 210dddae-a5ab-4c1f-83c0-09299a1e8ab6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Peticion del cliente: Viaje a Islandia, 5 dias, 2 personas, 2200 EUR en total. Lo unico que de verdad    │
│  importa es ver auroras boreales y banarnos en fuentes termales; el resto (vuelos, alojamiento) lo he revisado  │
│  ya manualmente.                                                                                                │
│                                                                                                                 │
│  Decide que especialistas necesitas y en que orden, dado las prioridades del cliente. Entrega un itinerario     │
│  completo y el presupuesto desglosado.                                                                          │
│  ID: eb546cb4-4e15-449a-ba4c-0cf9763590d5                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: Peticion del cliente: Viaje a Islandia, 5 dias, 2 personas, 2200 EUR en total. Lo unico que de verdad    │
│  importa es ver auroras boreales y banarnos en fuentes termales; el resto (vuelos, alojamiento) lo he revisado  │
│  ya manualmente.                                                                                                │
│                                                                                                                 │
│  Decide que especialistas necesitas y en que orden, dado las prioridades del cliente. Entrega un itinerario     │
│  completo y el presupuesto desglosado.                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#3) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'task': 'Crear un itinerario detallado para un viaje de 5 días a Islandia, centrado en la               │
│  visualización de auroras boreales y termales, incluyendo vuelos, alojamiento, actividades y transporte.',      │
│  'co...                                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Itinerario Detallado para un Viaje de 5 Días a Islandia                                                    │
│                                                                                                                 │
│  **Presupuesto Total: 2200 EUR para dos personas**                                                              │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  #### **Día 1: Llegada a Reykjavik**                                                                            │
│                                                                                                                 │
│  - **Transporte desde el Aeropuerto de Keflavik a Reykjavik**                                                   │
│    - Opción: Bus de traslado (Flybus)                                                                           │
│    - Costo: 30 EUR (15 EUR por persona)                                                                         │
│                                                                                                                 │
│  - **Actividad: Exploración de Reykjavik**                                                                      │
│    - Visitar Hallgrímskirkja y el puerto                                                                        │
│    - Costo: Gratuito                                                                                            │
│                                                                                                                 │
│  - **Alojamiento en Reykjavik**                                                                                 │
│    - Opción: Hotel central (2 noches)                                                                           │
│    - Costo: 200 EUR (100 EUR por noche)                                                                         │
│                                                                                                                 │
│  - **Cena en un restaurante local**                                                                             │
│    - Costo: 60 EUR (30 EUR por persona)                                                                         │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  #### **Día 2: Tour para Ver Auroras Boreales**                                                                 │
│                                                                                                                 │
│  - **Desayuno en el hotel**                                                                                     │
│    - Costo: Incluido                                                                                            │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Peticion del cliente: Viaje a Islandia, 5 dias, 2 personas, 2200 EUR en total. Lo unico que de verdad    │
│  importa es ver auroras boreales y banarnos en fuentes termales; el resto (vuelos, alojamiento) lo he revisado  │
│  ya manualmente.                                                                                                │
│                                                                                                                 │
│  Decide que especialistas necesitas y en que orden, dado las prioridades del cliente. Entrega un itinerario     │
│  completo y el presupuesto desglosado.                                                                          │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Itinerario Detallado para un Viaje de 5 Días a Islandia

**Presupuesto Total: 2200 EUR para dos personas**

---

#### **Día 1: Llegada a Reykjavik**

- **Transporte desde el Aeropuerto de Keflavik a Reykjavik**
  - Opción: Bus de traslado (Flybus)
  - Costo: 30 EUR (15 EUR por persona)
  
- **Actividad: Exploración de Reykjavik**
  - Visitar Hallgrímskirkja y el puerto
  - Costo: Gratuito
  
- **Alojamiento en Reykjavik**
  - Opción: Hotel central (2 noches)
  - Costo: 200 EUR (100 EUR por noche)
  
- **Cena en un restaurante local**
  - Costo: 60 EUR (30 EUR por persona)

---

#### **Día 2: Tour para Ver Auroras Boreales**

- **Desayuno en el hotel**
  - Costo: Incluido

- **Actividad: Tour de Auroras Boreales**
  - Operador: Reykjavik Excursions
  - Costo: 120 EUR (60 EUR por persona)
  
- **Cena en un restaurante local**
  - Costo: 60 EUR (30 EUR por persona)

---

#### **Día 3: Baño en Fuentes Termales**

- **Desayuno en el hotel**
  - Costo: Incluido

- **Actividad: Excursión 

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 210dddae-a5ab-4c1f-83c0-09299a1e8ab6                                                                       │
│  Final Output: ### Itinerario Detallado para un Viaje de 5 Días a Islandia                                      │
│                                                                                                                 │
│  **Presupuesto Total: 2200 EUR para dos personas**                                                              │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  #### **Día 1: Llegada a Reykjavik**                                                                            │
│                                                                                                                 │
│  - **Transporte desde el Aeropuerto de Keflavik a Reykjavik**                                                   │
│    - Opción: Bus de traslado (Flybus)                                                                           │
│    - Costo: 30 EUR (15 EUR por persona)                                                                         │
│                                                                                                                 │
│  - **Actividad: Exploración de Reykjavik**                                                                      │
│    - Visitar Hallgrímskirkja y el puerto                                                                        │
│    - Costo: Gratuito                                                                                            │
│                                                                                                                 │
│  - **Alojamiento en Reykjavik**                                                                                 │
│    - Opción: Hotel central (2 noches)                                                                           │
│    - Costo: 200 EUR (100 EUR por noche)                                                                         │
│                                                                                                                 │
│  - **Cena en un restaurante local**                                                                             │
│    - Costo: 60 EUR (30 EUR por persona)                                                                         │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  #### **Día 2: Tour para Ver Auroras Boreales**                                                                 │
│                                                                                                                 │
│  - **Desayuno en el hotel**                                                                                     │
│    - Costo: Incluido                                                                                            │
│                                                       

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Qué define este patrón

El manager decide en runtime cuánto delega a cada agente. Puede consultar poco a transporte y volver dos veces a actividades si la petición lo requiere.